# Coherent State Benchmark

This notebook validates the split-operator propagator against an exact analytic result.

A **coherent state** is a displaced harmonic oscillator ground state — the closest quantum
analogue to a classical oscillating particle. It has the special property that its centre
of mass follows the classical equation of motion *exactly*:

$$\langle x \rangle(t) = x_0 \cos(\omega t)$$

This is not an approximation — it holds for all time, regardless of $x_0$. It makes
an ideal benchmark: if our propagator is correct, the numerical $\langle x \rangle(t)$
must match this formula to within the expected $\mathcal{O}(\Delta t^2)$ error.

We also verify second-order convergence by showing that halving $\Delta t$ reduces
the error by a factor of ~4.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from attopy.propagators.splitop import SplitOperatorPropagator

# Use a clean matplotlib style
plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

## 1. Setup

All quantities are in **atomic units** (au) throughout.

We use a fine grid matched to the harmonic oscillator length scale (~1 au),
with a box large enough to contain the oscillating wavepacket comfortably.

In [ ]:
# --- Grid parameters ---
N   = 256       # grid points
dx  = 0.05      # spatial step (au)
dt  = 0.01      # time step (au)
L   = N * dx    # box size: 12.8 au

xgrid = np.linspace(-L/2, L/2, N, endpoint=False)

# --- Harmonic oscillator potential V(x) = x²/2, ω=1 au ---
omega = 1.0
V = 0.5 * omega**2 * xgrid**2

# --- Coherent state: ground state displaced to x0 ---
x0 = 2.0   # displacement (au) — stays well within box

psi0 = (omega / np.pi)**0.25 * np.exp(-0.5 * omega * (xgrid - x0)**2)
psi0 = psi0.astype(complex)
psi0 /= np.sqrt(np.sum(np.abs(psi0)**2) * dx)   # normalise

print(f"Box size:         {L:.1f} au")
print(f"Grid points:      {N}")
print(f"Grid spacing dx:  {dx} au")
print(f"Time step dt:     {dt} au")
print(f"Initial norm:     {np.sum(np.abs(psi0)**2) * dx:.10f}")
print(f"Initial <x>:      {np.sum(xgrid * np.abs(psi0)**2) * dx:.6f} au  (expected {x0})")

## 2. Propagation

We propagate for exactly two full oscillation periods $T = 2\pi/\omega$.

The `tgrid` spacing must match `dt` exactly — the propagator takes one step
of size `dt` per tgrid interval.

In [ ]:
T_period = 2.0 * np.pi / omega
n_periods = 2
T_total = n_periods * T_period

# Build tgrid with spacing exactly dt
n_steps = int(round(T_total / dt))
tgrid = np.linspace(0.0, n_steps * dt, n_steps + 1)

print(f"Propagation time: {T_total:.3f} au  ({n_periods} periods)")
print(f"Number of steps:  {n_steps}")

prop = SplitOperatorPropagator(dx=dx, dt=dt, xgrid=xgrid)
result = prop.propagate(psi0, tgrid, V.astype(complex), pulse=lambda t: 0.0)

print(f"Wall time:        {result.info['wall_time_s']:.3f} s")
print(f"Final norm:       {result.norm[-1]:.10f}")

## 3. Results: Dipole vs Analytic

In [ ]:
x_analytic = x0 * np.cos(omega * tgrid)
error = result.dipole - x_analytic
max_error = np.max(np.abs(error))

fig, axes = plt.subplots(2, 1, figsize=(9, 6),
                          gridspec_kw={'height_ratios': [3, 1]},
                          sharex=True)

# Mark period boundaries
for n in range(1, n_periods + 1):
    for ax in axes:
        ax.axvline(n * T_period, color='gray', lw=0.8, ls='--', alpha=0.5)

# Top panel: dipole vs analytic
ax = axes[0]
ax.plot(tgrid, x_analytic, 'k-', lw=1.5, label='Analytic: $x_0\cos(\omega t)$')
ax.plot(tgrid, result.dipole, 'C0--', lw=1.2, label='Split-operator (attopy)')
ax.set_ylabel(r'$\langle x \rangle$ (au)')
ax.legend(frameon=False)
ax.set_title(
    f'Coherent state oscillation  '
    f'($x_0={x0}$ au, $\omega={omega}$ au, $\Delta t={dt}$ au)',
    fontsize=11
)

# Bottom panel: error
ax = axes[1]
ax.plot(tgrid, error, 'C1-', lw=1.0)
ax.axhline(0, color='k', lw=0.6)
ax.set_ylabel('Error (au)')
ax.set_xlabel('Time (au)')
ax.set_title(f'Max error: {max_error:.2e} au', fontsize=10)

plt.tight_layout()
plt.savefig('coherent_state_dipole.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Max |error|: {max_error:.2e} au")

## 4. Norm Conservation

In [ ]:
norm_deviation = result.norm - 1.0

fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(tgrid, norm_deviation, 'C2-', lw=1.0)
ax.axhline(0, color='k', lw=0.6)
ax.set_xlabel('Time (au)')
ax.set_ylabel(r'$\|\psi\|^2 - 1$')
ax.set_title(f'Norm conservation  (max deviation: {np.max(np.abs(norm_deviation)):.2e})')
plt.tight_layout()
plt.show()

## 5. Wavepacket Snapshots

The probability density $|\psi(x,t)|^2$ at quarter-period intervals,
showing the wavepacket oscillating without spreading (a hallmark of coherent states).

In [ ]:
# Quarter-period snapshot times within the first period
snapshot_fractions = [0, 0.25, 0.5, 0.75, 1.0]
snapshot_labels = ['$t=0$', '$t=T/4$', '$t=T/2$', '$t=3T/4$', '$t=T$']
colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(snapshot_fractions)))

fig, ax = plt.subplots(figsize=(9, 4))

for frac, label, color in zip(snapshot_fractions, snapshot_labels, colors):
    t_target = frac * T_period
    idx = int(round(t_target / dt))
    density = np.abs(result.psi[idx])**2
    ax.plot(xgrid, density, color=color, lw=1.5, label=label)

ax.set_xlabel('Position (au)')
ax.set_ylabel(r'$|\psi(x,t)|^2$ (au$^{-1}$)')
ax.set_title('Probability density at quarter-period snapshots')
ax.legend(frameon=False, ncol=5, loc='upper right')
ax.set_xlim(-L/2, L/2)
plt.tight_layout()
plt.show()

## 6. Convergence Study

The Strang splitting has global error $\mathcal{O}(\Delta t^2)$: halving $\Delta t$
should reduce the error by a factor of ~4. We verify this by comparing the dipole
at $t = T$ for several values of $\Delta t$.

In [ ]:
T_conv = 1.6
x_exact = x0 * np.cos(omega * T_conv)

dt_values = [0.16, 0.08, 0.04, 0.02]
errors = []

for dt_c in dt_values:
    n = int(round(T_conv / dt_c))
    tg = np.linspace(0.0, n * dt_c, n + 1)
    p = SplitOperatorPropagator(dx=dx, dt=dt_c, xgrid=xgrid)
    r = p.propagate(psi0, tg, V.astype(complex), pulse=lambda t: 0.0)
    errors.append(abs(r.dipole[-1] - x_exact))
    print(f"dt = {dt_c:.4f} au   error = {errors[-1]:.2e} au")

# Fit slope in log-log space
log_dt = np.log10(dt_values)
log_err = np.log10(errors)
slope, intercept = np.polyfit(log_dt, log_err, 1)
print(f"\nFitted convergence order: {slope:.2f}  (expected 2.00)")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

ax.loglog(dt_values, errors, 'o-', color='C0', lw=1.5, ms=6, label='Measured error')

# Reference O(dt^2) line through the last point
dt_ref = np.array([dt_values[0], dt_values[-1]])
err_ref = errors[-1] * (dt_ref / dt_values[-1])**2
ax.loglog(dt_ref, err_ref, 'k--', lw=1.0, label=r'$\mathcal{O}(\Delta t^2)$ reference')

ax.set_xlabel(r'$\Delta t$ (au)')
ax.set_ylabel(r'$|\langle x\rangle(T) - x_{\rm exact}|$ (au)')
ax.set_title(f'Convergence study  (fitted order: {slope:.2f})')
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig('coherent_state_convergence.png', dpi=150, bbox_inches='tight')
plt.show()